# Phase 4 — Full Evaluation: Rule-Based + LLM-as-Judge Evaluators

## PURPOSE:
Runs the FoodPlanner agent against the Langfuse dataset and applies ALL evaluators in a single experiment run:

**Rule-based (from Phase 3):**
- `required_tools` — All expected tools were called
- `output_present` — Agent produced a non-empty answer
- `shopping_list` — Shopping list present when required

**LLM-as-judge (new in Phase 4, using create_llm_as_judge_evaluator):**
- `dietary_compliance` — Recipe respects all stated dietary restrictions
- `time_constraint` — Recipe fits within the stated time limit
- `recipe_present` — Output contains a concrete recipe
- `actionable_steps` — Instructions are clear enough to follow
- `serving_size` — Recipe serves the correct number of people

## WHAT TO CHECK AFTER RUNNING:
1. Terminal shows experiment summary with 8 score columns
2. Langfuse > Datasets > FoodPlannerEval shows a new experiment run
3. Each item has scores for all 8 metrics
4. Compare LLM judge scores vs rule-based scores for consistency


## Setup & Imports

In [1]:
import asyncio
import logging
import sys
from pathlib import Path
from typing import Any

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
)
logger = logging.getLogger(__name__)

# PATH SETUP
# Script lives at:  EVALUATION/phase4/run_evaluation.ipynb
PHASE4_DIR = Path.cwd()
PHASE3_DIR = PHASE4_DIR.parent / "phase3"
EVAL_AGENTS_ROOT = PHASE4_DIR.parents[1]
RUBRICS_DIR = PHASE4_DIR / "rubrics"

for p in [str(PHASE3_DIR), str(EVAL_AGENTS_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)


## Constants & Prompt Template

The default template passes the entire `output` dict to the judge.
We override it to pass only the recipe text (`final_output`) so
the judge focuses on language quality, not the internal tools dict.
We still include the tools_called list in the Expected Output section
so the judge has trajectory context when needed (e.g., cfia check).

In [2]:
DEFAULT_DATASET_NAME = "FoodPlannerEval"
DEFAULT_EXPERIMENT_NAME = "food_planner_v1_full_eval"
DEFAULT_MAX_CONCURRENCY = 3

FOOD_PLANNER_USER_PROMPT = """\
# User Query (Input)
{input}

# Expected Constraints (Expected Output)
{expected_output}

# Agent Recipe Response (Candidate Output to Evaluate)
{output}
"""

def _recipe_only_output(output: Any) -> str:
    """Extract only the final_output text for LLM judge prompts.

    The judge should see the recipe text, not the internal tools dict.
    For context, we also append the tools_called list.
    """
    if not isinstance(output, dict):
        return str(output)
    final = output.get("final_output", "")
    tools = output.get("tools_called", [])
    return (
        f"Recipe Text:\n{final}\n\n"
        f"Tools Called (in order): {tools}"
    )

def _make_recipe_prompt(output: Any, expected_output: Any, input: Any) -> str:
    """Build the user prompt for LLM judge calls."""
    from aieng.agent_evals.evaluation.graders._utils import serialize_for_prompt
    return FOOD_PLANNER_USER_PROMPT.format(
        input=serialize_for_prompt(input),
        expected_output=serialize_for_prompt(expected_output),
        output=_recipe_only_output(output),
    )


## LLM Judge Evaluators
Create all 4 LLM-as-judge evaluators from rubric files and wrap them to substitute the full output dict with recipe text.

In [3]:
def _build_llm_judges() -> list:
    """Create all 4 LLM-as-judge evaluators from rubric files."""
    from aieng.agent_evals.evaluation.graders import create_llm_as_judge_evaluator
    from aieng.agent_evals.evaluation.graders.config import LLMRequestConfig

    judge_config = LLMRequestConfig(temperature=0.0, retry_max_attempts=3)

    dietary = create_llm_as_judge_evaluator(
        name="dietary_compliance",
        model_config=judge_config,
        prompt_template=FOOD_PLANNER_USER_PROMPT,
        rubric_markdown=RUBRICS_DIR / "dietary_compliance.md",
    )

    time_limit = create_llm_as_judge_evaluator(
        name="time_constraint",
        model_config=judge_config,
        prompt_template=FOOD_PLANNER_USER_PROMPT,
        rubric_markdown=RUBRICS_DIR / "time_constraint.md",
    )

    completeness = create_llm_as_judge_evaluator(
        name="recipe_completeness",
        model_config=judge_config,
        prompt_template=FOOD_PLANNER_USER_PROMPT,
        rubric_markdown=RUBRICS_DIR / "recipe_completeness.md",
    )

    serving = create_llm_as_judge_evaluator(
        name="serving_size",
        model_config=judge_config,
        prompt_template=FOOD_PLANNER_USER_PROMPT,
        rubric_markdown=RUBRICS_DIR / "serving_size.md",
    )

    return [dietary, time_limit, completeness, serving]

def _wrap_judge_with_recipe_output(judge_fn):
    """Wrap an LLM judge to substitute the full output dict with recipe text."""
    async def wrapped(
        *,
        input: Any,  # noqa: A002
        output: Any,
        expected_output: Any,
        metadata: dict[str, Any] | None = None,
        **kwargs: Any,
    ):
        # Replace output dict with human-readable recipe text
        recipe_output = _recipe_only_output(output)
        return await judge_fn(
            input=input,
            output=recipe_output,
            expected_output=expected_output,
            metadata=metadata,
            **kwargs,
        )
    wrapped.__name__ = judge_fn.__name__
    return wrapped


## Run Evaluation
Run FoodPlanner full evaluation (rule-based + LLM-as-judge).

In [4]:
async def _main(
    dataset_name: str,
    experiment_name: str,
    max_concurrency: int,
) -> None:
    # ── Deferred imports (needs path setup above) ──
    from aieng.agent_evals.evaluation.experiment import run_experiment
    from aieng.agent_evals.async_client_manager import AsyncClientManager

    # Import rule-based evaluators and task from Phase 3
    try:
        from run_evaluation import (  # noqa: PLC0415
            FoodPlannerTask,
            required_tools_evaluator,
            output_present_evaluator,
            shopping_list_evaluator,
        )
    except ImportError as exc:
        logger.error(
            "Could not import from Phase 3. "
            "Ensure EVALUATION/phase3/run_evaluation.py exists in Coder: %s", exc
        )
        return

    print("\n=== Phase 4: FoodPlanner Full Evaluation (Rule-Based + LLM-as-Judge) ===")
    print(f"  Dataset      : {dataset_name}")
    print(f"  Experiment   : {experiment_name}")
    print(f"  Concurrency  : {max_concurrency}")
    print(f"  Rubrics from : {RUBRICS_DIR}")
    print("")

    # ── 1. Verify rubric files exist ──
    rubric_files = [
        "dietary_compliance.md",
        "time_constraint.md",
        "recipe_completeness.md",
        "serving_size.md",
    ]
    missing_rubrics = [f for f in rubric_files if not (RUBRICS_DIR / f).exists()]
    if missing_rubrics:
        logger.error(
            "Missing rubric files in %s: %s\n"
            "Paste the rubrics/ folder from EVALUATION/phase4/rubrics/ into Coder.",
            RUBRICS_DIR,
            missing_rubrics,
        )
        return
    logger.info("All 4 rubric files found.")

    # ── 2. Set up task ──
    task = FoodPlannerTask()
    try:
        task.setup()
    except Exception as exc:
        logger.error("Task setup failed: %s", exc)
        return

    # ── 3. Build evaluator list ──
    raw_llm_judges = _build_llm_judges()
    wrapped_judges = [_wrap_judge_with_recipe_output(j) for j in raw_llm_judges]

    all_evaluators = [
        # Rule-based (fast, deterministic)
        required_tools_evaluator,
        output_present_evaluator,
        shopping_list_evaluator,
        # LLM-as-judge (slow, semantic)
        *wrapped_judges,
    ]

    logger.info(
        "Running experiment with %d evaluators: %s",
        len(all_evaluators),
        [fn.__name__ for fn in all_evaluators],
    )

    # ── 4. Run experiment ──
    try:
        result = run_experiment(
            dataset_name,
            name=experiment_name,
            description=(
                "Phase 4 full evaluation — FoodPlanner agent on 30 ground truth queries. "
                "3 rule-based + 5 LLM-as-judge evaluators "
                "(dietary_compliance, time_constraint, recipe_present, "
                "actionable_steps, serving_size)."
            ),
            task=task.run,
            evaluators=all_evaluators,
            max_concurrency=max_concurrency,
            metadata={
                "phase": "4",
                "agent": "FoodPlanner",
                "evaluator_types": "rule_based+llm_judge",
                "rubrics_version": "v1",
            },
        )
    except Exception as exc:
        logger.error("run_experiment failed: %s", exc)
        return

    # ── 5. Print summary ──
    print("\n" + "=" * 65)
    print("  Experiment complete.")
    print(f"\n{result.format()}")
    print("\n  Score legend:")
    print("    required_tools     → all expected tools called (rule-based)")
    print("    output_present     → agent produced ≥50 char answer (rule-based)")
    print("    shopping_list      → shopping list present when needed (rule-based)")
    print("    dietary_compliance → recipe respects dietary restrictions (LLM)")
    print("    time_constraint    → recipe fits time limit (LLM)")
    print("    recipe_present     → output contains a concrete recipe (LLM)")
    print("    actionable_steps   → instructions are clear enough to follow (LLM)")
    print("    serving_size       → recipe serves the correct number (LLM)")
    print("\n  -> Open Langfuse > Datasets > FoodPlannerEval")
    print(f"     to inspect the '{experiment_name}' run.")
    print("  -> Green light: proceed to Phase 5 (analysis + final report).")
    print("=" * 65 + "\n")

    # ── 6. Close client ──
    try:
        await AsyncClientManager.get_instance().close()
    except Exception:
        pass


In [ ]:
# Run the evaluation in the notebook event loop
await _main(
    dataset_name=DEFAULT_DATASET_NAME,
    experiment_name=DEFAULT_EXPERIMENT_NAME,
    max_concurrency=DEFAULT_MAX_CONCURRENCY
)


2026-03-26 15:52:34,357 INFO __main__: All 4 rubric files found.



=== Phase 4: FoodPlanner Full Evaluation (Rule-Based + LLM-as-Judge) ===
  Dataset      : FoodPlannerEval
  Experiment   : food_planner_v1_full_eval
  Concurrency  : 3
  Rubrics from : /home/coder/eval-agents/EVALUATION/phase4/rubrics



2026-03-26 15:52:46,099 INFO run_evaluation: FoodPlannerTask ready (model=gemini-2.5-pro)
2026-03-26 15:52:46,137 INFO __main__: Running experiment with 7 evaluators: ['required_tools_evaluator', 'output_present_evaluator', 'shopping_list_evaluator', 'dietary_compliance', 'time_constraint', 'recipe_completeness', 'serving_size']
2026-03-26 15:52:46,608 WARNING langfuse: Propagated attribute 'experiment_item_metadata' value is over 200 characters (214 chars). Dropping value.
2026-03-26 15:52:46,726 WARNING langfuse: Propagated attribute 'experiment_item_metadata' value is over 200 characters (220 chars). Dropping value.
2026-03-26 15:52:56,089 INFO google_genai.models: AFC is enabled with max remote calls: 10.
2026-03-26 15:53:04,050 INFO google_genai.models: AFC is enabled with max remote calls: 10.
2026-03-26 15:53:07,443 INFO google_genai.models: AFC is enabled with max remote calls: 10.
2026-03-26 15:53:25,768 INFO google_genai.models: AFC is enabled with max remote calls: 10.
2026-